# F-S-N-SUPP-ARC Sensor-Noise Arc Geometry Baseline

Single-frequency sensor-noise support-recovery simulation with 3 microphones and 10 candidate sources restricted to a 10 degree sector. The notebook saves a geometry plot for verification and a coefficient MSE plot in dB versus sensor SNR.


In [ ]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "cs_priors").exists():
            return path
    raise RuntimeError("Could not find repository root containing src/cs_priors")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/cs_priors_matplotlib")

from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from cs_priors.plotting.plot_geometry import plot_sim_geometry
from cs_priors.simulation.mixing_model import quick_frequency_sim
from cs_priors.solvers.freq_lasso import frequency_lasso_solve
from cs_priors.solvers.freq_group_lasso import frequency_group_lasso_solve
from cs_priors.solvers.freq_sparse_prior import sparse_prior_solve


In [ ]:
TAG = "TEMP-F-S-N-SUPP-ARC"
# FIGURE_DIR = REPO_ROOT / "results" / "benchmarks" / TAG
# FIGURE_DIR.mkdir(parents=True, exist_ok=True)


# def save_figure(fig, stem: str):
#     fig.savefig(FIGURE_DIR / f"{stem}.pdf", dpi=300, bbox_inches="tight")


SNR_DB = np.array([10, 15, 20, 30, 40, 50, 60])
SEEDS = np.arange(5)

NUM_MICS = 3
NUM_SOURCES = 20
NUM_ACTIVE = 2
FREQUENCY_HZ = 100.0
SAMPLING_RATE = 2000.0
DURATION = 0.05
COMPONENT_AMPLITUDE = 1.0

MIC_RADIUS = 0.3
SOURCE_DISTANCE = 1.5
SECTOR_DEG = 360
SECTOR_RAD = np.deg2rad(SECTOR_DEG)
MIC_ANGLE_START = 0
MIC_ANGLE_SPAN = SECTOR_RAD
SOURCE_ANGLE_START = 0
SOURCE_ANGLE_SPAN = SECTOR_RAD

LASSO_ALPHA = 1e-4
GROUP_LASSO_ALPHA = 1e-4
LASSO_MAX_ITER = 5_000
GROUP_LASSO_MAX_ITER = 500
PLOT_FLOOR = 1e-16
GROUPING = "frequency"

METHOD_ORDER = ["r-LASSO", "Group LASSO", "Sparse Prior", "X_pinv"]
METHOD_COLORS = {
    "X_pinv": "tab:blue",
    "r-LASSO": "tab:orange",
    "Group LASSO": "tab:green",
    "Sparse Prior": "tab:red",
}
METHOD_LINESTYLES = {
    "X_pinv": ":",
    "r-LASSO": "-",
    "Group LASSO": "-",
    "Sparse Prior": "-",
}


In [ ]:
def make_simulation(snr_db: float, seed: int):
    return quick_frequency_sim(
        num_mics=NUM_MICS,
        num_sources=NUM_SOURCES,
        num_active=NUM_ACTIVE,
        seed=int(seed),
        sampling_rate=SAMPLING_RATE,
        duration=DURATION,
        source_distance=SOURCE_DISTANCE,
        mic_radius=MIC_RADIUS,
        array_type="arc",
        mic_angle_start=MIC_ANGLE_START,
        mic_angle_span=MIC_ANGLE_SPAN,
        source_angle_start=SOURCE_ANGLE_START,
        source_angle_span=SOURCE_ANGLE_SPAN,
        component_amplitude=COMPONENT_AMPLITUDE,
        magnitude_jitter=0.0,
        single_frequency_hz=FREQUENCY_HZ,
        sensor_snr_db=float(snr_db),
        model_snr_db=None,
        inverse_method="mp",
    )


def solve_methods(sim) -> dict[str, np.ndarray]:
    X_pinv = sim.X_pinv
    return {
        "X_pinv": X_pinv,
        "r-LASSO": frequency_lasso_solve(
            sim.Y, sim.A, alpha=LASSO_ALPHA, max_iter=LASSO_MAX_ITER, X_start=X_pinv
        ),
        "Group LASSO": frequency_group_lasso_solve(
            sim.Y,
            sim.A,
            alpha=GROUP_LASSO_ALPHA,
            grouping=GROUPING,
            max_iter=GROUP_LASSO_MAX_ITER,
            X_start=X_pinv,
        ),
        "Sparse Prior": sparse_prior_solve(X_pinv, sim.A, grouping=GROUPING),
    }


def mean_squared_difference(X_hat: np.ndarray, X_true: np.ndarray) -> float:
    return float(np.mean(np.abs(X_hat - X_true) ** 2))


def mse_to_db(values):
    return 10.0 * np.log10(np.maximum(np.asarray(values, dtype=float), PLOT_FLOOR))


def assert_simulation_contract(sim):
    assert sim.A.shape == (NUM_MICS, NUM_SOURCES, 1)
    assert np.allclose(sim.freqs, [FREQUENCY_HZ])
    assert len(sim.active_indices) == NUM_ACTIVE
    assert np.allclose(sim.delta, 0.0)
    assert np.linalg.norm(sim.eta) > 0.0


def run_case(snr_db: float, seed: int) -> list[dict]:
    sim = make_simulation(snr_db, seed)
    assert_simulation_contract(sim)

    rows = []
    for method, X_hat in solve_methods(sim).items():
        rows.append(
            {
                "tag": TAG,
                "sensor_snr_db": float(snr_db),
                "seed": int(seed),
                "method": method,
                "mean_squared_difference": mean_squared_difference(X_hat, sim.X),
            }
        )
    return rows


In [ ]:
geometry_sim = make_simulation(snr_db=60.0, seed=0)
assert_simulation_contract(geometry_sim)
fig, ax = plot_sim_geometry(geometry_sim, show=False)
ax.set_title(f"Arc geometry: M={NUM_MICS}, S={NUM_SOURCES}, sector={SECTOR_DEG:g} deg")
# save_figure(fig, "simulation_geometry")
# set dpi
fig.set_dpi(600)
plt.show()


In [ ]:
from cs_priors.plotting.plot_complex import plot_matrices
print(geometry_sim.A[:,:,0])
plot_matrices([geometry_sim.A[:,:,0].real, geometry_sim.A[:,:,0].imag], titles=["real A", "imag A"], dpi=300, font_size=6, show_values=False)
plot_matrices([geometry_sim.A[:,:,0]], dpi=300, font_size=2)

In [ ]:
rows = []
cases = product(SNR_DB, SEEDS)
total = len(SNR_DB) * len(SEEDS)

for snr_db, seed in tqdm(cases, total=total, desc="Running simulations"):
    rows.extend(run_case(float(snr_db), int(seed)))

results_df = pd.DataFrame(rows)
results_df["method"] = pd.Categorical(results_df["method"], categories=METHOD_ORDER, ordered=True)
results_df = results_df.sort_values(["sensor_snr_db", "seed", "method"]).reset_index(drop=True)
results_df.head()


In [ ]:
summary_df = (
    results_df
    .groupby(["sensor_snr_db", "method"], observed=False)
    .agg(
        median_mean_squared_difference=("mean_squared_difference", "median"),
        q25_mean_squared_difference=("mean_squared_difference", lambda x: x.quantile(0.25)),
        q75_mean_squared_difference=("mean_squared_difference", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

summary_df["median_mean_squared_difference_db"] = mse_to_db(summary_df["median_mean_squared_difference"])
summary_df["q25_mean_squared_difference_db"] = mse_to_db(summary_df["q25_mean_squared_difference"])
summary_df["q75_mean_squared_difference_db"] = mse_to_db(summary_df["q75_mean_squared_difference"])
summary_df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.4))

for method in METHOD_ORDER:
    method_df = summary_df[summary_df["method"] == method].sort_values("sensor_snr_db")
    x = method_df["sensor_snr_db"].to_numpy(dtype=float)
    median = method_df["median_mean_squared_difference_db"].to_numpy(dtype=float)
    q25 = method_df["q25_mean_squared_difference_db"].to_numpy(dtype=float)
    q75 = method_df["q75_mean_squared_difference_db"].to_numpy(dtype=float)

    ax.plot(
        x,
        median,
        marker="o",
        label=method,
        color=METHOD_COLORS[method],
        linestyle=METHOD_LINESTYLES[method],
    )
    ax.fill_between(x, q25, q75, color=METHOD_COLORS[method], alpha=0.18, linewidth=0)

ax.set_xticks(SNR_DB)
ax.set_xlabel("Sensor SNR (dB)")
ax.set_ylabel("Mean squared difference (dB)")
ax.set_title(f"Arc-sector coefficient MSE vs. sensor SNR (M={NUM_MICS}, S={NUM_SOURCES})")
ax.grid(True, linewidth=0.5, alpha=0.35)
ax.legend(frameon=False)

# save_figure(fig, "mean_squared_difference_db_vs_sensor_snr")
plt.show()


## Interpretation

This notebook isolates a difficult local-sector geometry: 3 microphones and 10 candidate sources inside a 10 degree sector. The MSE curve shows coefficient reconstruction error in dB as sensor noise decreases. The geometry plot verifies that both microphone and source placements are restricted to the same arc sector.
